# Lesson 17 — Conic Optimization

## Learning objectives

1. State the standard form of LP, SOCP, SDP and the chain of inclusions.
2. Recognize problems naturally formulated as SOCP (norm constraints, robust LP, portfolio).
3. Recognize SDP applications (LMIs, max-cut, polynomial optimization).
4. Use `discopt`'s SOCP backend (or any external SDP solver) on a small example.

## 1. Cones

A cone $K \subseteq \mathbb{R}^n$ is closed under non-negative scaling. The relevant cones for optimization {cite:p}`BenTal2001,Boyd2004`:

- **Linear cone** $\mathbb{R}^n_{\ge 0}$ — the LP cone.
- **Lorentz cone** (second-order cone) $\{(t, x) : t \ge \|x\|_2\}$.
- **PSD cone** $\mathbb{S}_+^n = \{X = X^\top, X \succeq 0\}$.

A *conic program*:

$$\min c^\top x \;\;\text{s.t.}\;\; Ax = b, \; x \in K.$$

LP, SOCP, SDP are conic programs over their respective cones. All three are convex; all three are polynomial-time via interior-point methods.

## 2. SOCP

$$\min c^\top x \;\;\text{s.t.}\;\; \|A_i x + b_i\|_2 \le c_i^\top x + d_i, \; i = 1, \dots, m.$$

Common applications {cite:p}`BenTal2001`:

- **Robust LP** with ellipsoidal uncertainty becomes an SOCP (Lesson 25).
- **Markowitz portfolio**: minimize $\|L^\top w\|_2$ where $\Sigma = LL^\top$.
- **Truss design**, antenna design, geometric medians.

QP and convex QCQP are special cases of SOCP.

## 3. SDP

$$\min \langle C, X \rangle \;\;\text{s.t.}\;\; \langle A_i, X \rangle = b_i, \; X \succeq 0.$$

Generalizes both LP (diagonal $X$) and SOCP (block structure). Applications:

- **Max-cut relaxation** (Goemans-Williamson): cut value $\le$ SDP value $\le$ trivial UB.
- **LMI control problems**: stability, $H_\infty$ control.
- **Polynomial optimization**: via sum-of-squares (Lasserre hierarchy).

SDPs are polynomial-time, but constants are large; solvers (Mosek, SDPT3, SeDuMi, COSMO) are slower than LP solvers.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model, sqrt
# (For sum-of-squares write `dm.sum(x**2)` — there is no separate `sum_squares` in discopt.)

# Markowitz: min sigma^2  s.t.  mu' w >= r_min, sum(w)=1, w>=0
n = 6
rng = np.random.default_rng(0)
mu = 0.05 + 0.1 * rng.random(n)              # expected returns
L  = rng.normal(scale=0.05, size=(n, n))     # covariance factor
Sigma = L @ L.T + 1e-3 * np.eye(n)

m = Model("portfolio")
w = m.continuous("w", shape=(n,), lb=0)
m.subject_to(dm.sum(w) == 1)
m.subject_to(mu @ w >= 0.08)
m.minimize(w @ Sigma @ w)                    # convex QP / SOCP
r = m.solve()
print("portfolio:", r.value(w).round(3), "  variance:", r.objective)


## 4. The conic chain

$$\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}.$$

Choosing the *smallest* cone that fits your problem is best — solvers exploit structure. A QP solved as an SDP is far slower than as an SOCP {cite:p}`BenTal2001`.

## 5. Strong duality and slater

Conic strong duality requires Slater (strict feasibility). If primal-dual values disagree at infinity (Slater fails), special handling is needed.

## References
{cite:p}`BenTal2001,Boyd2004` are the textbooks; {cite:p}`Wright1997` for IPM theory.